# Fase 5 - Actividad 7: Configuración GPU en Nube y Tokenización Global (Google Colab)

> **[IMPORTANTE - ENTORNO NUBE]**
> Este notebook **no está diseñado para ejecutarse en tu computadora local**. Debes subir este archivo a Google Drive, abrirlo con **Google Colab** y asegurarte de ir a `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y seleccionar **GPU (T4 o superior)**.

El objetivo de esta fase es inicializar la conexión con el hardware acelerado, montar el ecosistema de Google Drive para la persistencia de datos, y aplicar la **Tokenización oficial de BETO** (el Transformer para Español) preparando los tensores para el modelo profundo de forma global sobre todo el corpus.

In [ ]:
# 1. Verificación de Hardware (GPU)
# Certificamos que Google Colab nos asignó una Tarjeta Gráfica
!nvidia-smi

## 1. Montaje del Entorno y Dependencias

Instalamos las librerías del ecosistema de HuggingFace (`transformers`, `datasets`) que no solemos tener en la máquina local, y conectamos el disco de Google Drive.

In [ ]:
!pip install transformers datasets accelerate

from google.colab import drive
drive.mount('/content/drive')

## 2. Definición de Rutas de Trabajo y Carga Masiva

**Instrucción:** Debes subir la carpeta `data/` (que contiene `train_val_set.csv` y `test_blind_set.csv`) a tu Google Drive, por ejemplo dentro de una carpeta llamada `ProyectoFinal_ML`.

Cargaremos el dataset completo preservando su desbalance natural. No aplicaremos ningún submuestreo aquí para garantizar que en las futuras fases de validación los pliegues de prueba mantengan la distribución realista del mundo.

In [ ]:
import os
import pandas as pd

# Modifica esta ruta según dónde hayas subido los datos en tu Drive
base_path = '/content/drive/MyDrive/ProyectoFinal_ML/data'
train_path = os.path.join(base_path, 'train_val_set.csv')

print(f"Cargando datos masivos desde: {train_path}")
df = pd.read_csv(train_path, sep=';')

# Para BETO usamos la columna con limpieza conservadora
df['texto_beto'] = df['texto_beto'].fillna('')

# Preservamos el dataset original y desbalanceado
df_beto = pd.DataFrame({'texto': df['texto_beto'], 'label': df['label']})

print(f"Dimensiones originales: {df_beto.shape}")
print(f"Distribución original preservada:\n{df_beto['label'].value_counts().to_dict()}")

## 3. Tokenización Global (BETO: dccuchile/bert-base-spanish-wwm-cased)

A diferencia del TF-IDF, el tokenizador de BETO no aprende del vocabulario de nuestro dataset, sino que utiliza un diccionario precongelado por la Universidad de Chile. Por lo tanto, podemos tokenizar globalmente los 45,784 registros de un solo golpe sin riesgo de Data Leakage.

Utilizaremos la versión `cased` para respetar las mayúsculas y la puntuación original.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

modelo_beto = "dccuchile/bert-base-spanish-wwm-cased"
tokenizer = AutoTokenizer.from_pretrained(modelo_beto)

# Convertimos el Pandas DataFrame al formato 'Dataset' de HuggingFace
hf_dataset = Dataset.from_pandas(df_beto)

# Función de tokenización masiva
def tokenize_function(examples):
    # Aplicamos truncamiento y padding a una longitud segura
    return tokenizer(examples['texto'], padding='max_length', truncation=True, max_length=128)

print("Iniciando tokenización masiva de todos los registros en Español...")
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

print("\nDataset Tokenizado Exitosamente:")
print(tokenized_dataset)

## 4. Exportación de Tensores en Apache Arrow

Guardaremos este dataset masivo y desbalanceado ya tokenizado en el propio Google Drive. En el Paso 8 (Prevención OOM), cargaremos directamente esta estructura optimizada en la GPU.

In [ ]:
output_tensor_path = os.path.join(base_path, 'beto_tokenized_dataset')

tokenized_dataset.save_to_disk(output_tensor_path)
print(f"Tensores globales exportados y resguardados en Google Drive: {output_tensor_path}")